In [ ]:
# Check if GPU is available
import torch

print("PyTorch version:", torch.__version__)
print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))
else:
    print("No GPU found. We can still continue, but GPU is better.")

In [ ]:
!pip install gdown
!pip install opencv-python
!pip install kaggle

In [ ]:
# Kaggle API setup removed for security

In [ ]:
!kaggle datasets list

In [ ]:
!kaggle datasets list -s "salient object detection"

In [ ]:
from torch.utils.data import random_split, DataLoader
!kaggle datasets download -d balraj98/duts-saliency-detection-dataset

In [ ]:
 #!unzip duts-saliency-detection-dataset.zip -d dataset

In [ ]:
import os

print(os.path.exists("dataset/DUTS-TR"))

In [ ]:
import os
import cv2
import matplotlib.pyplot as plt

# Paths
image_path = "dataset/DUTS-TR/DUTS-TR-Image"
mask_path = "dataset/DUTS-TR/DUTS-TR-Mask"

# Get image name
image_name = os.listdir(image_path)[0]

# Read image
img = cv2.imread(os.path.join(image_path, image_name))
img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

# Mask filename
mask_name = image_name.replace(".jpg", ".png")

# Read mask
mask = cv2.imread(os.path.join(mask_path, mask_name), 0)

# Plot
plt.figure(figsize=(10,5))

plt.subplot(1,2,1)
plt.imshow(img)
plt.title("Input Image")
plt.axis("off")

plt.subplot(1,2,2)
plt.imshow(mask, cmap='gray')
plt.title("Ground Truth Mask")
plt.axis("off")

plt.show()

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image

In [ ]:
import os
from PIL import Image
import torch
from torch.utils.data import Dataset
from torchvision import transforms

class SODDataset(Dataset):
    def __init__(self, image_dir, mask_dir, image_size=128, augment=False):
        self.image_dir = image_dir
        self.mask_dir = mask_dir
        self.image_size = image_size
        self.augment = augment
        self.images = sorted(os.listdir(image_dir))

        self.resize = transforms.Resize((image_size, image_size))
        self.to_tensor = transforms.ToTensor()

        self.color_jitter = transforms.ColorJitter(
            brightness=0.2,
            contrast=0.2,
            saturation=0.1
        )

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        image_name = self.images[idx]
        mask_name = image_name.replace(".jpg", ".png")

        image = Image.open(os.path.join(self.image_dir, image_name)).convert("RGB")
        mask = Image.open(os.path.join(self.mask_dir, mask_name)).convert("L")

        image = self.resize(image)
        mask = self.resize(mask)

        if self.augment:
            if torch.rand(1).item() > 0.5:
                image = transforms.functional.hflip(image)
                mask = transforms.functional.hflip(mask)

            image = self.color_jitter(image)

        image = self.to_tensor(image)
        mask = self.to_tensor(mask)

        return image, mask

In [ ]:
image_dir = "dataset/DUTS-TR/DUTS-TR-Image"
mask_dir = "dataset/DUTS-TR/DUTS-TR-Mask"

full_dataset = SODDataset(image_dir, mask_dir, image_size=128, augment=False)

total_size = len(full_dataset)
train_size = int(0.70 * total_size)
val_size = int(0.15 * total_size)
test_size = total_size - train_size - val_size

train_indices, val_indices, test_indices = random_split(
    range(total_size),
    [train_size, val_size, test_size],
    generator=torch.Generator().manual_seed(42)
)

train_dataset_full = SODDataset(image_dir, mask_dir, image_size=128, augment=True)
val_dataset_full = SODDataset(image_dir, mask_dir, image_size=128, augment=False)
test_dataset_full = SODDataset(image_dir, mask_dir, image_size=128, augment=False)

train_dataset = torch.utils.data.Subset(train_dataset_full, train_indices.indices)
val_dataset = torch.utils.data.Subset(val_dataset_full, val_indices.indices)
test_dataset = torch.utils.data.Subset(test_dataset_full, test_indices.indices)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

print("Train samples:", len(train_dataset))
print("Validation samples:", len(val_dataset))
print("Test samples:", len(test_dataset))

In [ ]:
import torch
import torch.nn as nn

class SODNet(nn.Module):
    def __init__(self):
        super(SODNet, self).__init__()

        # Encoder
        self.encoder = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        # Decoder
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(64, 32, kernel_size=2, stride=2),
            nn.ReLU(),

            nn.ConvTranspose2d(32, 16, kernel_size=2, stride=2),
            nn.ReLU(),

            nn.ConvTranspose2d(16, 1, kernel_size=2, stride=2),
            nn.Sigmoid()
        )

    def forward(self, x):
        x = self.encoder(x)
        x = self.decoder(x)
        return x

In [ ]:
model = SODNet()

sample_image, sample_mask = full_dataset[0]

sample_image = sample_image.unsqueeze(0)

output = model(sample_image)

print("Output shape:", output.shape)

In [ ]:
import torch.nn.functional as F
import torch.optim as optim

def iou_loss(pred, target, smooth=1e-6):
    pred = pred.view(-1)
    target = target.view(-1)

    intersection = (pred * target).sum()
    union = pred.sum() + target.sum() - intersection

    iou = (intersection + smooth) / (union + smooth)
    return 1 - iou

def combined_loss(pred, target):
    bce = F.binary_cross_entropy(pred, target)
    iou = iou_loss(pred, target)
    return bce + 0.5 * iou

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = SODNet().to(device)

optimizer = optim.Adam(model.parameters(), lr=1e-3)

print("Using device:", device)

In [ ]:
from tqdm import tqdm

def calculate_iou(pred, target, threshold=0.5, smooth=1e-6):
    pred = (pred > threshold).float()
    target = (target > threshold).float()

    intersection = (pred * target).sum()
    union = pred.sum() + target.sum() - intersection

    return ((intersection + smooth) / (union + smooth)).item()


num_epochs = 5
best_val_loss = float("inf")

for epoch in range(num_epochs):
    model.train()
    train_loss = 0

    for images, masks in tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs} - Training"):
        images = images.to(device)
        masks = masks.to(device)

        outputs = model(images)
        loss = combined_loss(outputs, masks)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    train_loss = train_loss / len(train_loader)

    model.eval()
    val_loss = 0
    val_iou = 0

    with torch.no_grad():
        for images, masks in tqdm(val_loader, desc=f"Epoch {epoch+1}/{num_epochs} - Validation"):
            images = images.to(device)
            masks = masks.to(device)

            outputs = model(images)
            loss = combined_loss(outputs, masks)

            val_loss += loss.item()
            val_iou += calculate_iou(outputs, masks)

    val_loss = val_loss / len(val_loader)
    val_iou = val_iou / len(val_loader)

    print(f"Epoch [{epoch+1}/{num_epochs}]")
    print(f"Train Loss: {train_loss:.4f}")
    print(f"Val Loss: {val_loss:.4f}")
    print(f"Val IoU: {val_iou:.4f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), "best_sod_model.pth")
        print("Best model saved!")

In [ ]:
import matplotlib.pyplot as plt

model.load_state_dict(torch.load("best_sod_model.pth"))
model.eval()

image, mask = test_dataset[0]

input_image = image.unsqueeze(0).to(device)

with torch.no_grad():
    prediction = model(input_image)

image_np = image.permute(1, 2, 0).cpu().numpy()
mask_np = mask.squeeze().cpu().numpy()
pred_np = prediction.squeeze().cpu().numpy()

plt.figure(figsize=(15, 5))

plt.subplot(1, 3, 1)
plt.imshow(image_np)
plt.title("Input Image")
plt.axis("off")

plt.subplot(1, 3, 2)
plt.imshow(mask_np, cmap="gray")
plt.title("Ground Truth Mask")
plt.axis("off")

plt.subplot(1, 3, 3)
plt.imshow(pred_np, cmap="gray")
plt.title("Predicted Mask")
plt.axis("off")

plt.show()

In [ ]:
def calculate_metrics(pred, target, threshold=0.5):
    pred = (pred > threshold).float()
    target = (target > threshold).float()

    TP = (pred * target).sum().item()
    FP = (pred * (1 - target)).sum().item()
    FN = ((1 - pred) * target).sum().item()

    precision = TP / (TP + FP + 1e-6)
    recall = TP / (TP + FN + 1e-6)
    f1 = 2 * precision * recall / (precision + recall + 1e-6)

    intersection = (pred * target).sum().item()
    union = pred.sum().item() + target.sum().item() - intersection

    iou = intersection / (union + 1e-6)

    return precision, recall, f1, iou

In [ ]:
model.eval()

total_precision = 0
total_recall = 0
total_f1 = 0
total_iou = 0

count = 0

with torch.no_grad():
    for images, masks in test_loader:
        images = images.to(device)
        masks = masks.to(device)

        outputs = model(images)

        precision, recall, f1, iou = calculate_metrics(outputs, masks)

        total_precision += precision
        total_recall += recall
        total_f1 += f1
        total_iou += iou

        count += 1

print("Precision:", total_precision / count)
print("Recall:", total_recall / count)
print("F1-Score:", total_f1 / count)
print("IoU:", total_iou / count)

In [ ]:
plt.figure(figsize=(20,5))

plt.subplot(1,4,1)
plt.imshow(image_np)
plt.title("Input Image")
plt.axis("off")

plt.subplot(1,4,2)
plt.imshow(mask_np, cmap="gray")
plt.title("Ground Truth")
plt.axis("off")

plt.subplot(1,4,3)
plt.imshow(pred_np, cmap="gray")
plt.title("Predicted Mask")
plt.axis("off")

plt.subplot(1,4,4)
plt.imshow(image_np)
plt.imshow(pred_np, cmap="jet", alpha=0.5)
plt.title("Overlay")
plt.axis("off")

plt.show()

In [ ]:
class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(DoubleConv, self).__init__()

        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(),

            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU()
        )

    def forward(self, x):
        return self.conv(x)


class MiniUNet(nn.Module):
    def __init__(self):
        super(MiniUNet, self).__init__()

        self.enc1 = DoubleConv(3, 32)
        self.pool1 = nn.MaxPool2d(2)

        self.enc2 = DoubleConv(32, 64)
        self.pool2 = nn.MaxPool2d(2)

        self.enc3 = DoubleConv(64, 128)
        self.pool3 = nn.MaxPool2d(2)

        self.bottleneck = DoubleConv(128, 256)

        self.up3 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        self.dec3 = DoubleConv(256, 128)

        self.up2 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.dec2 = DoubleConv(128, 64)

        self.up1 = nn.ConvTranspose2d(64, 32, kernel_size=2, stride=2)
        self.dec1 = DoubleConv(64, 32)

        self.final = nn.Conv2d(32, 1, kernel_size=1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        e1 = self.enc1(x)
        p1 = self.pool1(e1)

        e2 = self.enc2(p1)
        p2 = self.pool2(e2)

        e3 = self.enc3(p2)
        p3 = self.pool3(e3)

        b = self.bottleneck(p3)

        u3 = self.up3(b)
        u3 = torch.cat([u3, e3], dim=1)
        d3 = self.dec3(u3)

        u2 = self.up2(d3)
        u2 = torch.cat([u2, e2], dim=1)
        d2 = self.dec2(u2)

        u1 = self.up1(d2)
        u1 = torch.cat([u1, e1], dim=1)
        d1 = self.dec1(u1)

        return self.sigmoid(self.final(d1))

In [ ]:
improved_model = MiniUNet().to(device)
improved_optimizer = optim.Adam(improved_model.parameters(), lr=1e-3)

print(improved_model)

In [ ]:
num_epochs = 25
best_improved_val_loss = float("inf")

for epoch in range(num_epochs):
    improved_model.train()
    train_loss = 0

    for images, masks in tqdm(train_loader, desc=f"Improved Epoch {epoch+1}/{num_epochs} - Training"):
        images = images.to(device)
        masks = masks.to(device)

        outputs = improved_model(images)
        loss = combined_loss(outputs, masks)

        improved_optimizer.zero_grad()
        loss.backward()
        improved_optimizer.step()

        train_loss += loss.item()

    train_loss = train_loss / len(train_loader)

    improved_model.eval()
    val_loss = 0
    val_iou = 0

    with torch.no_grad():
        for images, masks in tqdm(val_loader, desc=f"Improved Epoch {epoch+1}/{num_epochs} - Validation"):
            images = images.to(device)
            masks = masks.to(device)

            outputs = improved_model(images)
            loss = combined_loss(outputs, masks)

            val_loss += loss.item()
            val_iou += calculate_iou(outputs, masks)

    val_loss = val_loss / len(val_loader)
    val_iou = val_iou / len(val_loader)

    print(f"Improved Epoch [{epoch+1}/{num_epochs}]")
    print(f"Train Loss: {train_loss:.4f}")
    print(f"Val Loss: {val_loss:.4f}")
    print(f"Val IoU: {val_iou:.4f}")

    if val_loss < best_improved_val_loss:
        best_improved_val_loss = val_loss
        torch.save(improved_model.state_dict(), "best_mini_unet_sod_model.pth")
        print("Best improved model saved!")

In [ ]:
improved_model.load_state_dict(torch.load("best_mini_unet_sod_model.pth"))
improved_model.eval()

total_precision = 0
total_recall = 0
total_f1 = 0
total_iou = 0
count = 0

with torch.no_grad():
    for images, masks in test_loader:
        images = images.to(device)
        masks = masks.to(device)

        outputs = improved_model(images)

        precision, recall, f1, iou = calculate_metrics(outputs, masks)

        total_precision += precision
        total_recall += recall
        total_f1 += f1
        total_iou += iou
        count += 1

print("Improved Precision:", total_precision / count)
print("Improved Recall:", total_recall / count)
print("Improved F1-Score:", total_f1 / count)
print("Improved IoU:", total_iou / count)

In [ ]:
from google.colab import files

uploaded = files.upload()

In [ ]:
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import time

uploaded_image = list(uploaded.keys())[0]

img = Image.open(uploaded_image).convert("RGB")
original_size = img.size

transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor()
])

input_tensor = transform(img).unsqueeze(0).to(device)

improved_model.eval()

start_time = time.time()

with torch.no_grad():
    pred_mask = improved_model(input_tensor)

end_time = time.time()
inference_time = end_time - start_time

pred_mask = pred_mask.squeeze().cpu().numpy()

pred_mask_img = Image.fromarray((pred_mask * 255).astype(np.uint8))
pred_mask_img = pred_mask_img.resize(original_size)

plt.figure(figsize=(15,5))

plt.subplot(1,3,1)
plt.imshow(img)
plt.title("Uploaded Image")
plt.axis("off")

plt.subplot(1,3,2)
plt.imshow(pred_mask_img, cmap="gray")
plt.title("Predicted Mask")
plt.axis("off")

plt.subplot(1,3,3)
plt.imshow(img)
plt.imshow(pred_mask_img, cmap="jet", alpha=0.5)
plt.title("Overlay")
plt.axis("off")

plt.show()

print(f"Inference time: {inference_time:.4f} seconds")

| Model                  | Precision | Recall |     F1 |    IoU |
| ---------------------- | --------: | -----: | -----: | -----: |
| Baseline (5 epochs)    |    0.5768 | 0.5938 | 0.5807 | 0.4126 |
| Improved (5 epochs)    |    0.6479 | 0.5000 | 0.5606 | 0.3932 |
| Improved (15 epochs)   |    0.6278 | 0.5774 | 0.5981 | 0.4295 |
| Mini U-Net (25 epochs) |    0.8766 | 0.8830 | 0.8788 | 0.7854 |


The final improved Mini U-Net model trained for 25 epochs achieved the best overall performance, with Precision = 0.8766, Recall = 0.8830, F1-score = 0.8788, and IoU = 0.7854. The addition of skip connections, batch normalization, dropout, and data augmentation significantly improved segmentation quality and produced much clearer saliency masks compared to the baseline CNN model.

In [73]:
!pip install gradio

In [74]:
!pip install gradio -q

import os
import torch
import torch.nn as nn
from torchvision import transforms
from PIL import Image
import numpy as np
import gradio as gr

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

print("Files:", os.listdir())

class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(DoubleConv, self).__init__()

        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(),

            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU()
        )

    def forward(self, x):
        return self.conv(x)


class MiniUNet(nn.Module):
    def __init__(self):
        super(MiniUNet, self).__init__()

        self.enc1 = DoubleConv(3, 32)
        self.pool1 = nn.MaxPool2d(2)

        self.enc2 = DoubleConv(32, 64)
        self.pool2 = nn.MaxPool2d(2)

        self.enc3 = DoubleConv(64, 128)
        self.pool3 = nn.MaxPool2d(2)

        self.bottleneck = DoubleConv(128, 256)

        self.up3 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        self.dec3 = DoubleConv(256, 128)

        self.up2 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.dec2 = DoubleConv(128, 64)

        self.up1 = nn.ConvTranspose2d(64, 32, kernel_size=2, stride=2)
        self.dec1 = DoubleConv(64, 32)

        self.final = nn.Conv2d(32, 1, kernel_size=1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        e1 = self.enc1(x)
        p1 = self.pool1(e1)

        e2 = self.enc2(p1)
        p2 = self.pool2(e2)

        e3 = self.enc3(p2)
        p3 = self.pool3(e3)

        b = self.bottleneck(p3)

        u3 = self.up3(b)
        u3 = torch.cat([u3, e3], dim=1)
        d3 = self.dec3(u3)

        u2 = self.up2(d3)
        u2 = torch.cat([u2, e2], dim=1)
        d2 = self.dec2(u2)

        u1 = self.up1(d2)
        u1 = torch.cat([u1, e1], dim=1)
        d1 = self.dec1(u1)

        return self.sigmoid(self.final(d1))

model_path = "best_mini_unet_sod_model.pth"

if not os.path.exists(model_path):
    raise FileNotFoundError("best_mini_unet_sod_model.pth nuk ekziston në Colab. Upload-o file-in në Files panel.")

improved_model = MiniUNet().to(device)
improved_model.load_state_dict(torch.load(model_path, map_location=device))
improved_model.eval()

print("Model loaded successfully!")


def predict_saliency(input_image):
    original_image = input_image.convert("RGB")
    original_size = original_image.size

    transform = transforms.Compose([
        transforms.Resize((128, 128)),
        transforms.ToTensor()
    ])

    input_tensor = transform(original_image).unsqueeze(0).to(device)

    with torch.no_grad():
        pred_mask = improved_model(input_tensor)

    pred_mask = pred_mask.squeeze().cpu().numpy()
    pred_mask_img = Image.fromarray((pred_mask * 255).astype(np.uint8))
    pred_mask_img = pred_mask_img.resize(original_size)

    original_np = np.array(original_image).astype(np.float32)
    mask_np = np.array(pred_mask_img).astype(np.float32) / 255.0

    overlay = original_np.copy()
    overlay[:, :, 0] = overlay[:, :, 0] * 0.5 + 255 * mask_np * 0.5
    overlay = np.clip(overlay, 0, 255).astype(np.uint8)

    return pred_mask_img, Image.fromarray(overlay)


interface = gr.Interface(
    fn=predict_saliency,
    inputs=gr.Image(type="pil", label="Upload Image"),
    outputs=[
        gr.Image(label="Predicted Mask"),
        gr.Image(label="Overlay")
    ],
    title="Salient Object Detection App",
    description="Upload an image and the Mini U-Net model will generate a saliency mask."
)

interface.launch(share=True)

Device: cuda
Files: ['.config', '.gradio', 'best_sod_model.pth', 'images.jfif', 'dataset', 'best_mini_unet_sod_model.pth', 'duts-saliency-detection-dataset.zip', 'sample_data']
Model loaded successfully!
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://ec5ce5a113ee29c583.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
